In [0]:
from pyspark.sql.functions import col, sha2, lit, when, count, concat_ws, to_date
import traceback


In [0]:
storage_account_name = dbutils.secrets.get(scope="kv-finbank", key="datalake-account-name")
storage_account_access_key = dbutils.secrets.get(scope="kv-finbank", key="datalake-access-key")
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_access_key
)
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"

In [0]:

print("Ejecutando Control de Observabilidad: Anomalía de Volumen...")

# 1. Contar registros del batch actual
volumen_actual = df_mov_bronze.count()

# 2. Calcular promedio histórico (simulado de los últimos 7 días con los metadatos de Delta)
try:
    # Agrupamos por la fecha de ingesta para obtener el promedio de registros diarios
    df_historico = spark.read.format("delta").load(delta_bronze_path)
    promedio_7_dias = df_historico.groupBy(to_date("ingest_timestamp")).agg(count("*").alias("vol_diario")) \
        .selectExpr("avg(vol_diario)").collect()[0][0]
    
    # Si es la primera ejecución, evitamos errores matemáticos
    promedio_7_dias = promedio_7_dias if promedio_7_dias is not None else volumen_actual

    # 3. Calcular la desviación (Regla del 30%)
    if promedio_7_dias > 0:
        desviacion_porcentual = abs((volumen_actual - promedio_7_dias) / promedio_7_dias) * 100
        
        print(f"📊 Volumen actual: {volumen_actual} | Promedio 7 días: {promedio_7_dias:.0f}")
        print(f"📉 Desviación detectada: {desviacion_porcentual:.2f}%")
        
        if desviacion_porcentual > 30.0:
            alerta_msg = f"🚨 ALERTA DE VOLUMEN: La carga actual ({volumen_actual} filas) difiere más del 30% del promedio histórico ({promedio_7_dias:.0f})."
            print(alerta_msg)
            # Aquí se integraría una llamada a webhook/Slack. 
            # Para la prueba, lanzamos un error controlado para detener el pipeline como exige la rúbrica:
            raise ValueError(alerta_msg)
        else:
            print("✅ Control de volumen superado. El pipeline puede continuar.")

except Exception as e:
    # Capturamos la alerta para detener el proceso antes de la transformación
    print(f"🛑 Pipeline detenido por control de gobierno: {str(e)}")
    dbutils.notebook.exit("FAILED: VOLUMETRIC_ANOMALY")

In [0]:
from pyspark.sql.functions import col, sha2, lit, when, count
import traceback

print("Iniciando procesamiento avanzado Capa Silver...")
delta_bronze_path = f"{bronze_path}delta/"
error_path = f"{bronze_path}quarantine/" 
try:
    # 1. LECTURA DE LAS 6 TABLAS
    df_clientes = spark.read.format("delta").load(f"{delta_bronze_path}TB_CLIENTES_CORE")
    df_movimientos = spark.read.format("delta").load(f"{delta_bronze_path}TB_MOV_FINANCIEROS")
    df_productos = spark.read.format("delta").load(f"{delta_bronze_path}TB_PRODUCTOS_CAT")
    df_sucursales = spark.read.format("delta").load(f"{delta_bronze_path}TB_SUCURSALES_RED")
    df_oblig = spark.read.format("delta").load(f"{delta_bronze_path}TB_OBLIGACIONES")
    df_comisiones = spark.read.format("delta").load(f"{delta_bronze_path}TB_COMISIONES_LOG")

    # 2. LIMPIEZA: Clientes (PII y Nulos)
    df_clientes_clean = df_clientes.dropDuplicates().dropna(subset=["id_cli"]).fillna({"canal_adquis": "No Definido"})
    df_clientes_silver = df_clientes_clean \
        .withColumn("fec_nac", col("fec_nac").cast("date")) \
        .withColumn("num_doc_hash", sha2(col("num_doc").cast("string"), 256)) \
        .withColumn("nombre_hash", sha2(concat_ws(" ", col("nomb_cli"), col("apell_cli")), 256)) \
        .drop("num_doc", "nomb_cli", "apell_cli")

    # 3. LIMPIEZA: Movimientos e Integridad Referencial
    df_mov_clean = df_movimientos.dropDuplicates().filter(col("vr_mov") > 0)
    
    # Separar rechazados (los que no tienen cliente asociado) y los conformes
    df_mov_rechazados = df_mov_clean.join(df_clientes_silver, "id_cli", "left_anti") \
        .withColumn("motivo_error", lit("Violación Referencial: Cliente Inexistente"))
    df_mov_silver = df_mov_clean.join(df_clientes_silver, "id_cli", "left_semi")

    # 4. LIMPIEZA RESTANTE
    df_productos_silver = df_productos.dropDuplicates()
    df_sucursales_silver = df_sucursales.dropDuplicates()
    df_oblig_silver = df_oblig.dropDuplicates()
    df_comisiones_silver = df_comisiones.dropDuplicates()

    # 5. ESCRITURA EN CAPA SILVER Y CUARENTENA
    print("Escribiendo datos...")
    
    # Cuarentena
    df_mov_rechazados.write.format("delta").mode("append").save(f"{error_path}ERR_MOVIMIENTOS")
    
    # Tablas Silver
    df_clientes_silver.write.format("delta").mode("overwrite").save(f"{silver_path}TB_CLIENTES_CORE")
    df_mov_silver.write.format("delta").mode("overwrite").save(f"{silver_path}TB_MOV_FINANCIEROS")
    df_productos_silver.write.format("delta").mode("overwrite").save(f"{silver_path}TB_PRODUCTOS_CAT")
    df_sucursales_silver.write.format("delta").mode("overwrite").save(f"{silver_path}TB_SUCURSALES_RED")
    df_oblig_silver.write.format("delta").mode("overwrite").save(f"{silver_path}TB_OBLIGACIONES")
    df_comisiones_silver.write.format("delta").mode("overwrite").save(f"{silver_path}TB_COMISIONES_LOG")

    conformes = df_mov_silver.count()
    print(f"ÉXITO: {conformes} movimientos procesados y 6 tablas guardadas en Silver.")

except Exception as e:
    error_msg = f"Falla crítica en el pipeline Silver: {str(e)}"
    print(error_msg)
    traceback.print_exc()
    dbutils.notebook.exit(f"FAILED: {error_msg}")
else:
    dbutils.notebook.exit("SUCCESS")

In [0]:
print("Iniciando Pruebas Automatizadas de Calidad de Datos en Silver...\n")

# 1. Leer las tablas de la Capa Silver (ya procesadas)
df_clientes_dq = spark.read.format("delta").load(f"{silver_path}TB_CLIENTES_CORE")
df_movimientos_dq = spark.read.format("delta").load(f"{silver_path}TB_MOV_FINANCIEROS")

# Lista para almacenar los resultados
dq_results = []

# --- PRUEBA 1: Unicidad (Cero Clientes Duplicados) ---
total_clientes = df_clientes_dq.count()
clientes_unicos = df_clientes_dq.select("id_cli").distinct().count()
if total_clientes == clientes_unicos:
    dq_results.append(("1. Unicidad de Clientes", "✅ PASSED", "0 registros duplicados"))
else:
    dq_results.append(("1. Unicidad de Clientes", "❌ FAILED", f"{total_clientes - clientes_unicos} duplicados"))

# --- PRUEBA 2: Completitud (Cero Nulos en ID Cliente) ---
nulos_id = df_clientes_dq.filter(col("id_cli").isNull()).count()
if nulos_id == 0:
    dq_results.append(("2. Completitud de ID", "✅ PASSED", "0 valores nulos en id_cli"))
else:
    dq_results.append(("2. Completitud de ID", "❌ FAILED", f"{nulos_id} nulos encontrados"))

# --- PRUEBA 3: Regla de Negocio (Montos de Movimientos Positivos) ---
movs_invalidos = df_movimientos_dq.filter(col("vr_mov") <= 0).count()
if movs_invalidos == 0:
    dq_results.append(("3. Validez de Montos", "✅ PASSED", "100% movimientos > 0"))
else:
    dq_results.append(("3. Validez de Montos", "❌ FAILED", f"{movs_invalidos} montos inválidos"))

# --- PRUEBA 4: Integridad Referencial (Sin Movimientos Huérfanos) ---
huerfanos = df_movimientos_dq.join(df_clientes_dq, "id_cli", "left_anti").count()
if huerfanos == 0:
    dq_results.append(("4. Integridad Referencial", "✅ PASSED", "0 movimientos sin cliente asociado"))
else:
    dq_results.append(("4. Integridad Referencial", "❌ FAILED", f"{huerfanos} movimientos huérfanos"))

# --- PRUEBA 5: Consistencia de Fechas (Sin Clientes nacidos en el futuro) ---
fechas_invalidas = df_clientes_dq.filter(col("fec_nac") >= "2026-01-01").count()
if fechas_invalidas == 0:
    dq_results.append(("5. Consistencia de Fechas", "✅ PASSED", "0 fechas futuras detectadas"))
else:
    dq_results.append(("5. Consistencia de Fechas", "❌ FAILED", f"{fechas_invalidas} fechas inválidas"))


# --- GENERACIÓN DEL REPORTE VISUAL ---
print("=" * 65)
print("REPORTE DE CALIDAD DE DATOS (DATA OBSERVABILITY) - CAPA SILVER")
print("=" * 65)
print(f"{'NOMBRE DE LA PRUEBA'.ljust(30)} | {'ESTADO'.ljust(10)} | {'DETALLE'}")
print("-" * 65)
for test, status, detail in dq_results:
    print(f"{test.ljust(30)} | {status.ljust(10)} | {detail}")
print("=" * 65)

Iniciando Pruebas Automatizadas de Calidad de Datos en Silver...

REPORTE DE CALIDAD DE DATOS (DATA OBSERVABILITY) - CAPA SILVER
NOMBRE DE LA PRUEBA            | ESTADO     | DETALLE
-----------------------------------------------------------------
1. Unicidad de Clientes        | ✅ PASSED   | 0 registros duplicados
2. Completitud de ID           | ✅ PASSED   | 0 valores nulos en id_cli
3. Validez de Montos           | ✅ PASSED   | 100% movimientos > 0
4. Integridad Referencial      | ✅ PASSED   | 0 movimientos sin cliente asociado
5. Consistencia de Fechas      | ❌ FAILED   | 1 fechas inválidas


In [0]:
display(spark.read.format("delta").load(f"{bronze_path}quarantine/ERR_MOVIMIENTOS"))

id_cli,id_mov,cod_prod,num_cuenta,fec_mov,hra_mov,vr_mov,tip_mov,cod_canal,cod_ciudad,cod_estado_mov,id_dispositivo,ingest_timestamp,source_system,batch_id,ingest_year,ingest_month,ingest_day,motivo_error
FANTASMA_9999,MOV-0000001,PRD-0017,8746997995,2025-09-17T00:00:00,20:07:00,135223.64,Retiro,Web,13,Exitoso,45b263b7,2026-06-02T17:16:39.649161Z,SQL_FinBank_Core,BATCH_RUN_001,2026,6,2,Violación Referencial: Cliente Inexistente
